# OTV: a standalone walkthrough

This notebook demonstrates One-Token Verification with a tiny, randomly initialized Qwen3 model on CPU. It runs without downloading model weights and uses only the standalone `otv` package.

We will build a frozen reasoning cache, inspect the verification mask, check parallel probes against independent prefix runs, train on two toy traces, and save and reload the verifier. **These toy scores are not meaningful correctness estimates.** Real use requires the original reasoner and labeled traces.

From the repository root, install the notebook environment once:

```bash
python -m pip install -e ".[notebook]"
jupyter lab notebooks/otv_quickstart.ipynb
```

In [ ]:
from pathlib import Path
import json
import tempfile
import torch
from IPython.display import HTML, display
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast, Qwen3Config, Qwen3ForCausalLM
from otv import FrozenCache, OTVVerifier, make_targets
from otv.data import TraceCollator, TraceDataset, encode_trace
from otv.model import probe_mask
from otv.score import score_trace
from otv.train import train_step

torch.manual_seed(42)
torch.set_num_threads(1)
root = Path.cwd()
if not (root / "examples").is_dir():
    root = root.parent
assert (root / "examples/traces.example.jsonl").is_file(), "Run from the repository or notebooks directory."
print("Using CPU; no model weights will be downloaded.")

## 1. Create a local toy reasoner

The small vocabulary and two-layer model keep the example fast. Save the original model before attaching LoRA so reloads use exactly the same frozen reasoner.

In [ ]:
workspace = tempfile.TemporaryDirectory(prefix="otv-notebook-")
base_dir = Path(workspace.name) / "base"
vocab_words = ["[PAD]", "[BOS]", "[EOS]", "[UNK]", "Question", ":", "Response",
               "What", "is", "1", "+", "?", "2", "3", "=", ".", "Final", "answer"]
raw = Tokenizer(WordLevel({word: i for i, word in enumerate(vocab_words)}, unk_token="[UNK]"))
raw.pre_tokenizer = Whitespace()
tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw, pad_token="[PAD]",
                                   bos_token="[BOS]", eos_token="[EOS]", unk_token="[UNK]")
config = Qwen3Config(vocab_size=32, hidden_size=32, intermediate_size=64,
                     num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                     head_dim=8, max_position_embeddings=128, attention_dropout=0.0,
                     pad_token_id=0, bos_token_id=1, eos_token_id=2)
base = Qwen3ForCausalLM(config)
base.save_pretrained(base_dir)
tokenizer.save_pretrained(base_dir)
verifier = OTVVerifier.from_pretrained(base_dir, rank=2, alpha=4,
                                      head_dims=(16, 8), head_dropout=0).eval()
trainable = sum(p.numel() for p in verifier.parameters() if p.requires_grad)
print(f"Trainable verifier parameters: {trainable:,}")

## 2. Read two sources of K/V

A prefill with LoRA **disabled** creates detached reasoner K/V states. Verification enables LoRA and introduces one new truth-token query, key, and value per probe at each layer.

For a probe at prefix length `t`, the mask allows cached columns `0 .. t-1` and **only that probe's own new K/V column**. It blocks future cache columns and every other probe. Physical cache append indices and logical token positions are separate: the truth token's position ID is `t`.

In [ ]:
row = encode_trace(tokenizer, "What is 1 + 1?", "1 + 1 = 2.", prompt_format="plain")
ids = torch.tensor([row["input_ids"]])
cache = verifier.prefill(ids)
length = ids.shape[1]
positions = torch.tensor([[0, row["response_start"], length]])
mask = probe_mask(cache.attention_mask, positions, torch.float32)[0, 0]
allowed = mask == 0
headers = [f"K/V {i}" for i in range(length)] + [f"probe {i}" for i in range(3)]
html = '<div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:11px"><tr><th>Prefix t</th>'
html += ''.join(f'<th style="padding:4px">{h}</th>' for h in headers) + '</tr>'
for q, t in enumerate(positions[0].tolist()):
    html += f'<tr><th>{t}</th>'
    for column, visible in enumerate(allowed[q].tolist()):
        color = ("#75b798" if column < length else "#e9a363") if visible else "#e8e8e8"
        html += f'<td style="background:{color};padding:5px;text-align:center;color:#222">{int(visible)}</td>'
    html += '</tr>'
html += '</table><p>Green: frozen cache. Orange: new LoRA K/V. Gray: masked.</p></div>'
display(HTML(html))
print("Each layer's cache shape:", [tuple(k.shape) for k, _ in cache.layers])

## 3. Check parallel verification

The same frozen full-trace cache supports all prefix probes in one forward call. The comparison below independently prefills each shorter prefix and checks that it produces the same scores. Nonzero LoRA updates ensure the comparison exercises adapter gating as well as attention masking.

In [ ]:
with torch.no_grad():
    for name, parameter in verifier.lm.named_parameters():
        if "lora_B" in name:
            parameter.normal_(std=0.05)
    cached_before = [(k.clone(), v.clone()) for k, v in cache.layers]
    parallel = verifier(cache)
    independent = []
    for t in range(length + 1):
        if t:
            shorter = verifier.prefill(ids[:, :t])
        else:
            shorter = FrozenCache(tuple((k[:, :, :0], v[:, :, :0]) for k, v in cache.layers),
                                  cache.attention_mask[:, :0])
        independent.append(verifier(shorter, [t]))
    independent = torch.cat(independent, dim=1)
    torch.testing.assert_close(parallel, independent, atol=2e-6, rtol=1e-5)
assert all(torch.equal(k, old_k) and torch.equal(v, old_v)
           for (k, v), (old_k, old_v) in zip(cache.layers, cached_before))
print("Maximum parallel/single-probe difference:", (parallel - independent).abs().max().item())
print("The original reasoner cache is unchanged.")

## 4. Train from final outcome labels

For response progress `u` and final label `y`, the linear target is `0.5 + (y - 0.5) * u`. The standalone boundary convention includes the point before the first response token (`u=0`) and the complete response (`u=1`). Other rules are available through `make_targets` or `--target-mode`.

Only LoRA, the truth embedding, and the regression head receive gradients. Query chunking frees each chunk's computation graph after backward while reusing the detached cache.

In [ ]:
dataset = TraceDataset(root / "examples/traces.example.jsonl", tokenizer,
                       max_length=128, prompt_format="plain")
batch = TraceCollator(tokenizer.pad_token_id)([dataset[0], dataset[1]])
optimizer = torch.optim.AdamW((p for p in verifier.parameters() if p.requires_grad),
                              lr=0.01, weight_decay=0)
frozen_before = {n: p.detach().clone() for n, p in verifier.named_parameters() if not p.requires_grad}
losses = [train_step(verifier, batch, optimizer, chunk_size=4) for _ in range(12)]
assert all(torch.equal(p, frozen_before[n])
           for n, p in verifier.named_parameters() if n in frozen_before)
print("Toy training MSE:", [round(loss, 4) for loss in losses])
print("Base model weights stayed frozen.")

In [ ]:
verifier.eval()
question = "What is 1 + 1?"
for response in ["1 + 1 = 2. Final answer: 2.", "1 + 1 = 3. Final answer: 3."]:
    result = score_trace(verifier, tokenizer, question, response, chunk_size=4, prompt_format="plain")
    print(json.dumps({"response": response, "trace_score": round(result["trace_score"], 4),
                      "final_score": round(result["final_score"], 4)}))

## 5. Save and reload

The checkpoint stores the verification adapters, head, truth embedding, and tokenizer. The base weights remain separate. These standalone checkpoints use a different format from the original LLaMA-Factory integration.

In [ ]:
checkpoint = Path(workspace.name) / "verifier"
verifier.save_pretrained(checkpoint, tokenizer)
loaded = OTVVerifier.from_pretrained(checkpoint=checkpoint).eval()
reference = score_trace(verifier, tokenizer, question, "1 + 1 = 2.", prompt_format="plain")
restored = score_trace(loaded, tokenizer, question, "1 + 1 = 2.", prompt_format="plain")
torch.testing.assert_close(torch.tensor(reference["prefix_scores"]),
                           torch.tensor(restored["prefix_scores"]), atol=1e-7, rtol=0)
print("Checkpoint reload preserves all prefix scores.")
print("Saved files:", sorted(path.name for path in checkpoint.iterdir()))

## 6. Move to real traces

Use a full-attention Qwen3, Qwen2/Qwen2.5, or Llama reasoner. Keep the model, tokenizer, and prompt format consistent with trace generation. Prepare JSONL records containing `question`, `response`, and binary `correct`, then run from the repository root:

```bash
otv-train --model /path/to/base-model \
  --train-data /path/to/train_traces.jsonl \
  --eval-data /path/to/heldout_traces.jsonl \
  --output outputs/otv_run --device cuda:0 --dtype bfloat16 \
  --probe-chunk-size 128 --target-mode linear

otv-score --checkpoint outputs/otv_run \
  --input /path/to/test_traces.jsonl --output results/scores.jsonl \
  --device cuda:0
```

`--target-mode` also accepts `constant`, `sigmoid`, `exponential`, `wave`, `step`, and the exploratory `sine` option. Use a new output directory for each run. This single-device reference has CPU numerical checks; full-scale CUDA and paper benchmark results must be evaluated separately.

See the [README](../README.md), the [paper](https://aclanthology.org/2026.emnlp-main.519/), and the downloadable [interactive blog](../docs/index.html?raw=1) for further details.

In [ ]:
workspace.cleanup()
print("Removed the temporary toy weights and checkpoint.")